# Data Loading
load from existing activity summary excel

In [2]:
import pandas as pd
from importlib import reload
import numpy as np
import os
import sys
PROJECT_ROOT = (os.path.abspath(os.path.join(
                  os.path.dirname("..\\Streamlit_App\\")
                  )))
sys.path.append(PROJECT_ROOT)

from PDU_Func import IO_Data, Activity


c:\Users\irsya\Documents\Jiunx_Files\PROJECT\2021-PDU\01_PDU_python\Streamlit_App


'..'

In [3]:
initial_path = "C:\\Users\\irsya\\Documents\\Jiunx_Files\\PROJECT\\2021-PDU\\01_PDU_python\\Data\\Master_Report\\ToBeUploadDB\\fortest\\"

WellExcel_Dict = {
    "AA-03":initial_path + "AA-03.xlsx",
    "AAE-05":initial_path + "AAE-05.xlsx",
    "AAE-06":initial_path + "AAE-06.xlsx",
}
# 140
# 146
# 150
for i in WellExcel_Dict.values():
    print(i)

C:\Users\irsya\Documents\Jiunx_Files\PROJECT\2021-PDU\01_PDU_python\Data\Master_Report\ToBeUploadDB\fortest\AA-03.xlsx
C:\Users\irsya\Documents\Jiunx_Files\PROJECT\2021-PDU\01_PDU_python\Data\Master_Report\ToBeUploadDB\fortest\AAE-05.xlsx
C:\Users\irsya\Documents\Jiunx_Files\PROJECT\2021-PDU\01_PDU_python\Data\Master_Report\ToBeUploadDB\fortest\AAE-06.xlsx


In [4]:
Data_List = []

for i in WellExcel_Dict:
    Data_List.append(pd.read_excel(WellExcel_Dict[i], sheet_name="MERGE"))

for Data_Temp in Data_List:
    display(Data_Temp.head(5))
Data_Temp = Data_List[0]
    

,Date,Time,Date/Time,Duration,Hole Depth,Bit Depth,Meterage (m) (Drilling),Sub Activity,Activity,Remarks,section
0,2021-04-16,19:46:35,2021-04-16 19:46:35,87.000000,9.60,0.00,NaN,Connection,Trip In,"RIH 26"" Bit","26"""
1,2021-04-16,21:13:35,2021-04-16 21:13:35,7.666667,35.00,27.39,NaN,Moving,Trip In,"RIH 26"" Bit, starting spud @21:00","26"""
2,2021-04-16,21:21:15,2021-04-16 21:21:15,14.583333,35.00,27.19,NaN,Connection,Trip In,NaN,"26"""
3,2021-04-16,21:35:50,2021-04-16 21:35:50,11.416667,35.00,27.20,NaN,Reaming,Drilling Formation,Reaming prior on bottom,"26"""
4,2021-04-16,21:47:15,2021-04-16 21:47:15,55.833333,49.69,35.31,14.69,Rotary Drilling,Drilling Formation,Drilling of Formation @ 35 mMD,"26"""


,Date,Time,Date/Time,Duration,Hole Depth,Bit Depth,Meterage (m) (Drilling),Sub Activity,Activity,Remarks,section
0,2021-07-14,16:09:00,2021-07-14 16:09:00,272.583333,45.00,45.00,NaN,Rotary Drilling,Drilling Formation,NaN,"26"""
1,2021-07-14,20:41:35,2021-07-14 20:41:35,18.416667,63.83,18.83,NaN,reaming,Drilling Formation,"Stand Down, reaming 2 times","26"""
2,2021-07-14,21:00:00,2021-07-14 21:00:00,19.416667,63.83,NaN,18.83,Connection,Drilling Formation,NaN,"26"""
3,2021-07-14,21:19:25,2021-07-14 21:19:25,5.250000,63.83,63.83,NaN,reaming,Drilling Formation,NaN,"26"""
4,2021-07-14,21:24:40,2021-07-14 21:24:40,303.433333,92.65,92.65,28.82,Rotary Drilling,Drilling Formation,NaN,"26"""


,Date,Time,Date/Time,Duration,Hole Depth,Bit Depth,Meterage (m) (Drilling),Sub Activity,Activity,Remarks,section
0,2021-08-16,07:00:00,2021-08-16 07:00:00,22.366667,40.00,0.00,NaN,Other,Other,NaN,"26"""
1,2021-08-16,07:22:22,2021-08-16 07:22:22,409.000000,65.22,65.22,25.22,Rotary Drilling,Drilling Formation,NaN,"26"""
2,2021-08-16,14:11:22,2021-08-16 14:11:22,13.666667,65.22,65.22,NaN,Reaming,Drilling Formation,"Stand down, reaming 2 times","26"""
3,2021-08-16,14:25:02,2021-08-16 14:25:02,27.666667,64.92,64.92,NaN,Other,Other,TDS Problem,"26"""
4,2021-08-16,14:52:42,2021-08-16 14:52:42,16.000000,65.22,65.22,25.22,Connection,Drilling Formation,NaN,"26"""


# Data Transform & Preparation
0. rename and match the column name
1. clean repeated activity
2. generate duration
3. generate label


In [5]:
reload(IO_Data)
reload(Activity)

<module 'PDU_Func.Activity' from 'c:\\Users\\irsya\\Documents\\Jiunx_Files\\PROJECT\\2021-PDU\\01_PDU_python\\Streamlit_App\\PDU_Func\\Activity.py'>

In [6]:
list_col_rename = {"Date":"Date",
# "Time"
"Date/Time" : "Start Time",
"End Time" : "End Time",
"Duration": "Duration (Minutes)",
"Hole Depth" : "Hole Depth (Max)",
"Bit Depth":"Bit Depth(mean)",
"Meterage (m) (Drilling)":"Drilling Meterage (m)",
"Sub Activity":"SUB-ACTIVITY",
"Activity":"ACTIVITY",
"Remarks":"Remarks",
"section":"Section"
}
Data_Temp.rename(columns=list_col_rename, inplace=True)
Data_Temp["End Time"] = Data_Temp["Start Time"].shift(-1)
Data_Temp['ACTIVITY'] = Data_Temp['ACTIVITY'].str.upper()
Data_Temp['SUB-ACTIVITY'] = Data_Temp['SUB-ACTIVITY'].str.title()
Data_Temp['PIC'] = "MIH"
# Data_Temp = Data_Temp[list_col_rename.values()]
# create empty column
list_dest_col = ['STATUS', 'Date', 'Start Time', 'End Time', 'ACTIVITY', 'SUB-ACTIVITY', 
                'CONNECTION-ACTIVITY', 'Section', 'Duration (Minutes)', 'Hole Depth (Max)', 
                'Bit Depth(mean)', 'Drilling Meterage (m)', 'MERGE_SubActivity-Activity', 
                'Rotate Drilling Time (Minutes)', 'Slide Drilling Time (Minutes)', 
                'Reaming Time (Minutes)', 'Connection Time (Minutes)', 'Total Stand Drilling Meterage (m)', 
                'Total Stand Duration (hrs)', 'On Bottom state (hrs)', 'Stand Group', 'PIC', 'Remarks']

for dest_col in list_dest_col:
    # print(dest_col in Data_Temp.columns)
    if not (dest_col in Data_Temp.columns):
        Data_Temp[dest_col] = np.NaN
Data_Temp['Drilling Meterage (m)'] = Data_Temp['Drilling Meterage (m)'].astype('float')
Data_Temp = Activity.cleanRepeatedActivity(Data_Temp)
Data_Temp = Activity.labelStand_v3(Data_Temp)

Data_Temp

,STATUS,Date,Start Time,End Time,ACTIVITY,SUB-ACTIVITY,CONNECTION-ACTIVITY,Section,Duration (Minutes),Hole Depth (Max),...,Rotate Drilling Time (Minutes),Slide Drilling Time (Minutes),Reaming Time (Minutes),Connection Time (Minutes),Total Stand Drilling Meterage (m),Total Stand Duration (hrs),On Bottom state (hrs),Stand Group,PIC,Remarks
0,0.0,2021-04-16,2021-04-16 19:46:35,2021-04-16 21:13:35,TRIP IN,Connection,Connection--1,"26""",87.000000,9.60,...,0.000000,0.0,0.000000,87.000000,0.0,0.00,0.0,,MIH,"RIH 26"" Bit"
1,0.0,2021-04-16,2021-04-16 21:13:35,2021-04-16 21:21:15,TRIP IN,Moving,,"26""",7.666667,35.00,...,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--1,MIH,"RIH 26"" Bit, starting spud @21:00"
2,0.0,2021-04-16,2021-04-16 21:21:15,2021-04-16 21:35:50,TRIP IN,Connection,Connection--2,"26""",14.583333,35.00,...,0.000000,0.0,0.000000,14.583333,0.0,0.37,0.0,,MIH,0
3,0.0,2021-04-16,2021-04-16 21:35:50,2021-04-16 21:47:15,DRILLING FORMATION,Reaming,Post-Connection--2,"26""",11.416667,35.00,...,0.000000,0.0,11.416667,0.000000,0.0,0.00,0.0,Stand Group--2,MIH,Reaming prior on bottom
4,0.0,2021-04-16,2021-04-16 21:47:15,2021-04-16 22:43:05,DRILLING FORMATION,Rotary Drilling,,"26""",55.833333,49.69,...,55.833333,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--2,MIH,Drilling of Formation @ 35 mMD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1972,0.0,2021-05-11,2021-05-11 08:10:42,2021-05-11 08:12:22,TRIP OUT,Moving,,"12-1/2""",1.666667,1750.00,...,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--793,MIH,0
1973,0.0,2021-05-11,2021-05-11 08:12:22,2021-05-11 08:19:22,TRIP OUT,Connection,Connection--794,"12-1/2""",7.000000,1750.00,...,0.000000,0.0,0.000000,7.000000,0.0,0.14,0.0,,MIH,0
1974,0.0,2021-05-11,2021-05-11 08:19:22,2021-05-11 08:22:22,TRIP OUT,Moving,,"12-1/2""",3.000000,1750.00,...,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--794,MIH,0
1975,0.0,2021-05-11,2021-05-11 08:22:22,2021-05-11 08:25:00,TRIP OUT,Connection,Connection--795,"12-1/2""",2.633333,1750.00,...,0.000000,0.0,0.000000,2.633333,0.0,0.09,0.0,,MIH,0


In [37]:
reload(IO_Data)
# 146
# 150
wid = 150

IO_Data.UploadActivitySummary_v2(Data_Temp, wid)

add new data to Activity Summary
{'wid': 150, 'date': '2021-08-16', 'time_start': '2021-08-16 07:00:00', 'time_end': '2021-08-16 07:22:22', 'duration_minutes': 22.36666666693054, 'hole_depth': 40.0, 'bit_depth': 0.0, 'meterage_drilling': 0.0, 'rotate_drilling_time': 0.0, 'slide_drilling_time': 0.0, 'reaming_time': 0.0, 'connection_time': 0.0, 'on_bottom_hours': 0.0, 'stand_duration': 0.0, 'label_subactivity': 'Other', 'label_activity': 'OTHER', 'stand_meterage_drilling': 0.0, 'stand_durationx': '', 'pic': 'MIH', 'section': '26"', 'remark': '0', 'stand_group': '', 'stand_on_bottom': 0}
add new data to Activity Summary
{'wid': 150, 'date': '2021-08-16', 'time_start': '2021-08-16 07:22:22', 'time_end': '2021-08-16 14:11:22', 'duration_minutes': 409.00000000256114, 'hole_depth': 65.22, 'bit_depth': 65.22, 'meterage_drilling': 25.22, 'rotate_drilling_time': 409.00000000256114, 'slide_drilling_time': 0.0, 'reaming_time': 0.0, 'connection_time': 0.0, 'on_bottom_hours': 0.0, 'stand_duration': 

In [12]:
import requests
import json
TableAPI = "https://pdumitradome.id/dome_api/rtdc/ActivitySummary/get_data"
json_queries = json.dumps(
    {
    "wid": 150,
    "start" : start_date_time,
    "end" : end_date_time
    },
    indent = 4
)
response = (
    requests.get(
        TableAPI, data=json_queries
    )
)
        # print((response.json()))

JSON_DF = pd.DataFrame(dict(response.json())['result'])
JSON_DF

,date,time_start,time_end,duration_minutes,hole_depth,bit_depth,meterage_drilling,rotate_drilling_time,slide_drilling_time,reaming_time,connection_time,on_bottom_hours,stand_duration,label_subactivity,label_activity,stand_meterage_drilling,stand_durationx,stand_on_bottom,pic,section,remark,stand_group
0,2021-08-16,2021-08-16 07:00:00,2021-08-16 07:22:22,22.3667,40,0,0,0,0,0,0,0,0,Other,OTHER,0,0,0,MIH,"26""",0,
1,2021-08-16,2021-08-16 07:22:22,2021-08-16 14:11:22,409,65.22,65.22,25.22,409,0,0,0,0,0,Rotary Drilling,DRILLING FORMATION,0,0,0,MIH,"26""",0,
2,2021-08-16,2021-08-16 14:11:22,2021-08-16 14:25:02,13.6667,65.22,65.22,0,0,0,13.6667,0,0,0,Reaming,DRILLING FORMATION,0,0,0,MIH,"26""","Stand down, reaming 2 times",
3,2021-08-16,2021-08-16 14:25:02,2021-08-16 14:52:42,27.6667,64.92,64.92,0,0,0,0,0,0,0,Other,OTHER,0,0,0,MIH,"26""",TDS Problem,
4,2021-08-16,2021-08-16 14:52:42,2021-08-16 15:08:42,16,65.22,65.22,25.22,0,0,0,16,0,0,Connection,DRILLING FORMATION,0,0,0,MIH,"26""",0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2598,2021-09-17,2021-09-17 08:00:00,2021-09-17 11:00:00,180,2550,2544,0,0,0,0,0,0,0,Circulation,CIRCULATE HOLE CLEANING,0,0,0,MIH,"12-1/4""","Release stuck pipe, POOH DP from 1623.7 mMD",
2599,2021-09-17,2021-09-17 11:00:00,2021-09-17 12:30:00,90,1623,1623,0,0,0,0,0,0,0,Cementing Job,CEMENTING JOB,0,0,0,MIH,"12-1/4""",0,
2600,2021-09-17,2021-09-17 12:30:00,2021-09-17 16:20:00,230,1623,1623,0,0,0,0,0,0,0,Moving,TRIP OUT,0,0,0,MIH,"12-1/4""",0,
2601,2021-09-17,2021-09-17 16:20:00,2021-09-17 20:30:00,250,1623,1623,0,0,0,0,0,0,0,Wait On Cement,WAIT ON CEMENT,0,0,0,MIH,"12-1/4""",Slip and Cut Drilling line,


In [14]:
JSON_DF.to_excel('tes.xlsx')

In [9]:
reload(IO_Data)
pd.set_option("display.max_columns", 100)
start_date_time =  '2000-10-17 00:00:00'
end_date_time = '2050-10-17 00:00:00'


dataDome = IO_Data.DomeGetData_v2(150, start_date_time, end_date_time, table_type="Activity Summary")
dataDome

get Activity Summary data/table from DOME


ParserError: year 0 is out of range: 0000-00-00 00:00:00

In [15]:
start_time_del = "2021-09-17 20:30:00"
IO_Data.DomeDelete(150, start_time_del, table_type="Activity Summary")

delete dome of Activity Summary data/table from DOME


In [40]:
# for start_time_del in dataDome["Start Time"].astype('string').values:

#     IO_Data.DomeDelete(wid, start_time_del, table_type="Activity Summary")

delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table from DOME
delete dome of Activity Summary data/table fro

In [40]:
list_wid = [129,130,135,136,140,143,144,145,146,150,156,160,170,175,182,186]
for wid_create in list_wid:
    if IO_Data.DomeCekTable(wid_create)['status'] == 404:
        IO_Data.DomeCreateTable(wid_create)
    if IO_Data.DomeCekTable(wid_create, table_type='Activity Summary')['status'] == 404:
        IO_Data.DomeCreateTable(wid_create, table_type='Activity Summary')

In [16]:
Data_Temp

,STATUS,Date,Start Time,End Time,ACTIVITY,SUB-ACTIVITY,CONNECTION-ACTIVITY,Section,Duration (Minutes),Hole Depth (Max),Bit Depth(mean),Drilling Meterage (m),MERGE_SubActivity-Activity,Rotate Drilling Time (Minutes),Slide Drilling Time (Minutes),Reaming Time (Minutes),Connection Time (Minutes),Total Stand Drilling Meterage (m),Total Stand Duration (hrs),On Bottom state (hrs),Stand Group,PIC,Remarks
0,0.0,2021-04-16,2021-04-16 19:46:35,2021-04-16 21:13:35,TRIP IN,Connection,Connection--1,"26""",87.000000,9.60,0.00,0.00,TRIP IN--Connection,0.000000,0.0,0.000000,87.000000,0.0,0.00,0.0,,MIH,"RIH 26"" Bit"
1,0.0,2021-04-16,2021-04-16 21:13:35,2021-04-16 21:21:15,TRIP IN,Moving,,"26""",7.666667,35.00,27.39,0.00,TRIP IN--Moving,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--1,MIH,"RIH 26"" Bit, starting spud @21:00"
2,0.0,2021-04-16,2021-04-16 21:21:15,2021-04-16 21:35:50,TRIP IN,Connection,Connection--2,"26""",14.583333,35.00,27.19,0.00,TRIP IN--Connection,0.000000,0.0,0.000000,14.583333,0.0,0.37,0.0,,MIH,0
3,0.0,2021-04-16,2021-04-16 21:35:50,2021-04-16 21:47:15,DRILLING FORMATION,Reaming,Post-Connection--2,"26""",11.416667,35.00,27.20,0.00,DRILLING FORMATION--Reaming,0.000000,0.0,11.416667,0.000000,0.0,0.00,0.0,Stand Group--2,MIH,Reaming prior on bottom
4,0.0,2021-04-16,2021-04-16 21:47:15,2021-04-16 22:43:05,DRILLING FORMATION,Rotary Drilling,,"26""",55.833333,49.69,35.31,14.69,DRILLING FORMATION--Rotary Drilling,55.833333,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--2,MIH,Drilling of Formation @ 35 mMD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1972,0.0,2021-05-11,2021-05-11 08:10:42,2021-05-11 08:12:22,TRIP OUT,Moving,,"12-1/2""",1.666667,1750.00,88.49,0.00,TRIP OUT--Moving,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--793,MIH,0
1973,0.0,2021-05-11,2021-05-11 08:12:22,2021-05-11 08:19:22,TRIP OUT,Connection,Connection--794,"12-1/2""",7.000000,1750.00,7.90,0.00,TRIP OUT--Connection,0.000000,0.0,0.000000,7.000000,0.0,0.14,0.0,,MIH,0
1974,0.0,2021-05-11,2021-05-11 08:19:22,2021-05-11 08:22:22,TRIP OUT,Moving,,"12-1/2""",3.000000,1750.00,7.90,0.00,TRIP OUT--Moving,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--794,MIH,0
1975,0.0,2021-05-11,2021-05-11 08:22:22,2021-05-11 08:25:00,TRIP OUT,Connection,Connection--795,"12-1/2""",2.633333,1750.00,7.90,0.00,TRIP OUT--Connection,0.000000,0.0,0.000000,2.633333,0.0,0.09,0.0,,MIH,0


In [18]:
list(Data_Temp['SUB-ACTIVITY'].unique())

['Connection',
 'Moving',
 'Reaming',
 'Rotary Drilling',
 'Cementing',
 'Other',
 'Rig Repair',
 'Stationary',
 'Slide Drilling',
 'Cementing Job',
 'Wait On Cement',
 'Make Up Bha',
 'Circulation',
 'Pressure Test',
 'Lay Down Bha']

In [20]:
Data_Temp[['Start Time', 'End Time']].values

array([['2021-04-16T19:46:35.000000000', '2021-04-16T21:13:35.000000000'],
       ['2021-04-16T21:13:35.000000000', '2021-04-16T21:21:15.000000000'],
       ['2021-04-16T21:21:15.000000000', '2021-04-16T21:35:50.000000000'],
       ...,
       ['2021-05-11T08:19:22.000000000', '2021-05-11T08:22:22.000000000'],
       ['2021-05-11T08:22:22.000000000', '2021-05-11T08:25:00.000000000'],
       ['2021-05-11T08:25:00.000000000', '2021-05-11T10:47:00.000000000']],
      dtype='datetime64[ns]')

# Time2Depth Plot

In [60]:
import plotly.graph_objects as go
Data_test_1 = Data_Temp.reset_index()
Data_test_2 = Data_Temp.reset_index()
Data_test_3 = Data_Temp.reset_index()
display(Data_test_3)

Data_test_2['Start Time'] = Data_test_2['End Time']
Data_test_3['Start Time'] = Data_test_3['End Time']
Data_test_3['Bit Depth(mean)'] = Data_test_3['Bit Depth(mean)'].shift(-1)

Data_test = pd.concat([Data_test_1, Data_test_2, Data_test_3], ignore_index=True)

Data_test = Data_test.sort_values(by=['index', 'Start Time'])
display(Data_test[['Start Time','Bit Depth(mean)','SUB-ACTIVITY']])



list_subactivity_unique = list(Data_test['SUB-ACTIVITY'].unique())
figTimeDepth = go.Figure()



for subactivity_unique in list_subactivity_unique:
    TimeDepth_DF = Data_test.copy()
    # if subactivity_unique == "Rotary Drilling":
    TimeDepth_DF[TimeDepth_DF["SUB-ACTIVITY"] != subactivity_unique] = None
    figTimeDepth.add_trace(
        go.Scatter(
            x=TimeDepth_DF['Start Time'], 
            y=TimeDepth_DF["Bit Depth(mean)"],
            mode='lines',
            name=subactivity_unique,
            hovertemplate='Date: %{x} <br>Bit Depth: %{y}'
            )
        )
figTimeDepth.update_layout(
    xaxis_title="Date/Time",
    yaxis_title="Bit Depth(mean)",
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="Rockwell"
    ))
figTimeDepth['layout']['yaxis']['autorange'] = "reversed"
figTimeDepth.show()


,index,STATUS,Date,Start Time,End Time,ACTIVITY,SUB-ACTIVITY,CONNECTION-ACTIVITY,Section,Duration (Minutes),Hole Depth (Max),Bit Depth(mean),Drilling Meterage (m),MERGE_SubActivity-Activity,Rotate Drilling Time (Minutes),Slide Drilling Time (Minutes),Reaming Time (Minutes),Connection Time (Minutes),Total Stand Drilling Meterage (m),Total Stand Duration (hrs),On Bottom state (hrs),Stand Group,PIC,Remarks
0,0,0.0,2021-04-16,2021-04-16 19:46:35,2021-04-16 21:13:35,TRIP IN,Connection,Connection--1,"26""",87.000000,9.60,0.00,0.00,TRIP IN--Connection,0.000000,0.0,0.000000,87.000000,0.0,0.00,0.0,,MIH,"RIH 26"" Bit"
1,1,0.0,2021-04-16,2021-04-16 21:13:35,2021-04-16 21:21:15,TRIP IN,Moving,,"26""",7.666667,35.00,27.39,0.00,TRIP IN--Moving,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--1,MIH,"RIH 26"" Bit, starting spud @21:00"
2,2,0.0,2021-04-16,2021-04-16 21:21:15,2021-04-16 21:35:50,TRIP IN,Connection,Connection--2,"26""",14.583333,35.00,27.19,0.00,TRIP IN--Connection,0.000000,0.0,0.000000,14.583333,0.0,0.37,0.0,,MIH,0
3,3,0.0,2021-04-16,2021-04-16 21:35:50,2021-04-16 21:47:15,DRILLING FORMATION,Reaming,Post-Connection--2,"26""",11.416667,35.00,27.20,0.00,DRILLING FORMATION--Reaming,0.000000,0.0,11.416667,0.000000,0.0,0.00,0.0,Stand Group--2,MIH,Reaming prior on bottom
4,4,0.0,2021-04-16,2021-04-16 21:47:15,2021-04-16 22:43:05,DRILLING FORMATION,Rotary Drilling,,"26""",55.833333,49.69,35.31,14.69,DRILLING FORMATION--Rotary Drilling,55.833333,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--2,MIH,Drilling of Formation @ 35 mMD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1972,1972,0.0,2021-05-11,2021-05-11 08:10:42,2021-05-11 08:12:22,TRIP OUT,Moving,,"12-1/2""",1.666667,1750.00,88.49,0.00,TRIP OUT--Moving,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--793,MIH,0
1973,1973,0.0,2021-05-11,2021-05-11 08:12:22,2021-05-11 08:19:22,TRIP OUT,Connection,Connection--794,"12-1/2""",7.000000,1750.00,7.90,0.00,TRIP OUT--Connection,0.000000,0.0,0.000000,7.000000,0.0,0.14,0.0,,MIH,0
1974,1974,0.0,2021-05-11,2021-05-11 08:19:22,2021-05-11 08:22:22,TRIP OUT,Moving,,"12-1/2""",3.000000,1750.00,7.90,0.00,TRIP OUT--Moving,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--794,MIH,0
1975,1975,0.0,2021-05-11,2021-05-11 08:22:22,2021-05-11 08:25:00,TRIP OUT,Connection,Connection--795,"12-1/2""",2.633333,1750.00,7.90,0.00,TRIP OUT--Connection,0.000000,0.0,0.000000,2.633333,0.0,0.09,0.0,,MIH,0


,Start Time,Bit Depth(mean),SUB-ACTIVITY
0,2021-04-16 19:46:35,0.00,Connection
1977,2021-04-16 21:13:35,0.00,Connection
3954,2021-04-16 21:13:35,27.39,Connection
1,2021-04-16 21:13:35,27.39,Moving
1978,2021-04-16 21:21:15,27.39,Moving
...,...,...,...
3952,2021-05-11 08:25:00,7.90,Connection
5929,2021-05-11 08:25:00,0.00,Connection
1976,2021-05-11 08:25:00,0.00,Stationary
3953,2021-05-11 10:47:00,0.00,Stationary


# ON BOTTOM

,STATUS,Date,Start Time,End Time,ACTIVITY,SUB-ACTIVITY,CONNECTION-ACTIVITY,Section,Duration (Minutes),Hole Depth (Max),Bit Depth(mean),Drilling Meterage (m),MERGE_SubActivity-Activity,Rotate Drilling Time (Minutes),Slide Drilling Time (Minutes),Reaming Time (Minutes),Connection Time (Minutes),Total Stand Drilling Meterage (m),Total Stand Duration (hrs),On Bottom state (hrs),Stand Group,PIC,Remarks
0,0.0,2021-04-16,2021-04-16 19:46:35,2021-04-16 21:13:35,TRIP IN,Connection,Connection--1,"26""",87.000000,9.60,0.00,0.00,TRIP IN--Connection,0.000000,0.0,0.000000,87.000000,0.0,0.00,0.0,,MIH,"RIH 26"" Bit"
1,0.0,2021-04-16,2021-04-16 21:13:35,2021-04-16 21:21:15,TRIP IN,Moving,,"26""",7.666667,35.00,27.39,0.00,TRIP IN--Moving,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--1,MIH,"RIH 26"" Bit, starting spud @21:00"
2,0.0,2021-04-16,2021-04-16 21:21:15,2021-04-16 21:35:50,TRIP IN,Connection,Connection--2,"26""",14.583333,35.00,27.19,0.00,TRIP IN--Connection,0.000000,0.0,0.000000,14.583333,0.0,0.37,0.0,,MIH,0
3,0.0,2021-04-16,2021-04-16 21:35:50,2021-04-16 21:47:15,DRILLING FORMATION,Reaming,Post-Connection--2,"26""",11.416667,35.00,27.20,0.00,DRILLING FORMATION--Reaming,0.000000,0.0,11.416667,0.000000,0.0,0.00,0.0,Stand Group--2,MIH,Reaming prior on bottom
4,0.0,2021-04-16,2021-04-16 21:47:15,2021-04-16 22:43:05,DRILLING FORMATION,Rotary Drilling,,"26""",55.833333,49.69,35.31,14.69,DRILLING FORMATION--Rotary Drilling,55.833333,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--2,MIH,Drilling of Formation @ 35 mMD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1972,0.0,2021-05-11,2021-05-11 08:10:42,2021-05-11 08:12:22,TRIP OUT,Moving,,"12-1/2""",1.666667,1750.00,88.49,0.00,TRIP OUT--Moving,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--793,MIH,0
1973,0.0,2021-05-11,2021-05-11 08:12:22,2021-05-11 08:19:22,TRIP OUT,Connection,Connection--794,"12-1/2""",7.000000,1750.00,7.90,0.00,TRIP OUT--Connection,0.000000,0.0,0.000000,7.000000,0.0,0.14,0.0,,MIH,0
1974,0.0,2021-05-11,2021-05-11 08:19:22,2021-05-11 08:22:22,TRIP OUT,Moving,,"12-1/2""",3.000000,1750.00,7.90,0.00,TRIP OUT--Moving,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--794,MIH,0
1975,0.0,2021-05-11,2021-05-11 08:22:22,2021-05-11 08:25:00,TRIP OUT,Connection,Connection--795,"12-1/2""",2.633333,1750.00,7.90,0.00,TRIP OUT--Connection,0.000000,0.0,0.000000,2.633333,0.0,0.09,0.0,,MIH,0


In [68]:
OBH_ROP_DF = Data_Temp.copy()

OBH_ROP_DF = OBH_ROP_DF[OBH_ROP_DF['SUB-ACTIVITY']=='Connection']
# OBH_ROP_DF = OBH_ROP_DF.dropna(subset=['Stand Group_Pred_Shift'])
OBH_ROP_DF = OBH_ROP_DF[OBH_ROP_DF['Total Stand Drilling Meterage (m)']!=0]
OBH_ROP_DF['On Bottom ROP (m/hr)'] = OBH_ROP_DF['Total Stand Drilling Meterage (m)'] / (OBH_ROP_DF['On Bottom state (hrs)'])
OBH_ROP_DF['Avg. ROP Per Stand'] = OBH_ROP_DF['Total Stand Drilling Meterage (m)'] / (OBH_ROP_DF['Total Stand Duration (hrs)'])
OBH_ROP_DF
import plotly.express as px
fig = px.bar(OBH_ROP_DF, x="Start Time", y=['On Bottom ROP (m/hr)','Avg. ROP Per Stand'],
              barmode='group',
             height=400)
fig.show()

In [83]:
Data_Temp

,STATUS,Date,Start Time,End Time,ACTIVITY,SUB-ACTIVITY,CONNECTION-ACTIVITY,Section,Duration (Minutes),Hole Depth (Max),Bit Depth(mean),Drilling Meterage (m),MERGE_SubActivity-Activity,Rotate Drilling Time (Minutes),Slide Drilling Time (Minutes),Reaming Time (Minutes),Connection Time (Minutes),Total Stand Drilling Meterage (m),Total Stand Duration (hrs),On Bottom state (hrs),Stand Group,PIC,Remarks
0,0.0,2021-04-16,2021-04-16 19:46:35,2021-04-16 21:13:35,TRIP IN,Connection,Connection--1,"26""",87.000000,9.60,0.00,0.00,TRIP IN--Connection,0.000000,0.0,0.000000,87.000000,0.0,0.00,0.0,,MIH,"RIH 26"" Bit"
1,0.0,2021-04-16,2021-04-16 21:13:35,2021-04-16 21:21:15,TRIP IN,Moving,,"26""",7.666667,35.00,27.39,0.00,TRIP IN--Moving,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--1,MIH,"RIH 26"" Bit, starting spud @21:00"
2,0.0,2021-04-16,2021-04-16 21:21:15,2021-04-16 21:35:50,TRIP IN,Connection,Connection--2,"26""",14.583333,35.00,27.19,0.00,TRIP IN--Connection,0.000000,0.0,0.000000,14.583333,0.0,0.37,0.0,,MIH,0
3,0.0,2021-04-16,2021-04-16 21:35:50,2021-04-16 21:47:15,DRILLING FORMATION,Reaming,Post-Connection--2,"26""",11.416667,35.00,27.20,0.00,DRILLING FORMATION--Reaming,0.000000,0.0,11.416667,0.000000,0.0,0.00,0.0,Stand Group--2,MIH,Reaming prior on bottom
4,0.0,2021-04-16,2021-04-16 21:47:15,2021-04-16 22:43:05,DRILLING FORMATION,Rotary Drilling,,"26""",55.833333,49.69,35.31,14.69,DRILLING FORMATION--Rotary Drilling,55.833333,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--2,MIH,Drilling of Formation @ 35 mMD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1972,0.0,2021-05-11,2021-05-11 08:10:42,2021-05-11 08:12:22,TRIP OUT,Moving,,"12-1/2""",1.666667,1750.00,88.49,0.00,TRIP OUT--Moving,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--793,MIH,0
1973,0.0,2021-05-11,2021-05-11 08:12:22,2021-05-11 08:19:22,TRIP OUT,Connection,Connection--794,"12-1/2""",7.000000,1750.00,7.90,0.00,TRIP OUT--Connection,0.000000,0.0,0.000000,7.000000,0.0,0.14,0.0,,MIH,0
1974,0.0,2021-05-11,2021-05-11 08:19:22,2021-05-11 08:22:22,TRIP OUT,Moving,,"12-1/2""",3.000000,1750.00,7.90,0.00,TRIP OUT--Moving,0.000000,0.0,0.000000,0.000000,0.0,0.00,0.0,Stand Group--794,MIH,0
1975,0.0,2021-05-11,2021-05-11 08:22:22,2021-05-11 08:25:00,TRIP OUT,Connection,Connection--795,"12-1/2""",2.633333,1750.00,7.90,0.00,TRIP OUT--Connection,0.000000,0.0,0.000000,2.633333,0.0,0.09,0.0,,MIH,0


In [99]:


StandTimeBreakdown_DF = Data_Temp.groupby(['Stand Group']).agg({
            'Start Time':'first', 
            'Rotate Drilling Time (Minutes)':'sum', 
            'Slide Drilling Time (Minutes)':'sum',
            'Reaming Time (Minutes)':'sum',
            'Connection Time (Minutes)':'sum',
            }
        ).reset_index()
StandTimeBreakdown_DF = StandTimeBreakdown_DF[(StandTimeBreakdown_DF["Stand Group"].str.contains('Stand'))]

#     # StandTimeBreakdown_DF = StandTimeBreakdown_DF[ConnectionTime_DF['RotateDrilling'] != 0]
#     # StandTimeBreakdown_DF = StandTimeBreakdown_DF[ConnectionTime_DF['RotateDrilling'] != 0]
StandTimeBreakdown_DF = StandTimeBreakdown_DF.loc[~(StandTimeBreakdown_DF[['Rotate Drilling Time (Minutes)','Slide Drilling Time (Minutes)','Reaming Time (Minutes)']]==0).all(axis=1)]
display(StandTimeBreakdown_DF)
# # st.dataframe(StandTimeBreakdown_DF)

# SummaryActivity_DF['Stand Group_Pred_Shift'] = SummaryActivity_DF['Stand Group_Pred'].shift(1)

# # with st.expander('Stand Time Breakdown'):
figStandTimeBreakdown = px.bar(StandTimeBreakdown_DF, x='Start Time', y=['Rotate Drilling Time (Minutes)','Slide Drilling Time (Minutes)','Reaming Time (Minutes)'], width=1000, height=1000)
# figStandTimeBreakdown = px.bar(SummaryActivity_DF.dropna(subset=['Stand Group_Pred', "Duration(minutes)"]), x='Stand Group_Pred', y="Duration(minutes)", color="LABEL_SubActivity", width=1000, height=1000)
figStandTimeBreakdown.update_xaxes(type='category')
figStandTimeBreakdown.update_layout(
                # title="Title",
                xaxis=dict(
                    title="Stand Group"
                ),
                yaxis=dict(
                    title="Duration(Minutes)"
                ) ) 

figStandTimeBreakdown.show()


,Stand Group,Start Time,Rotate Drilling Time (Minutes),Slide Drilling Time (Minutes),Reaming Time (Minutes),Connection Time (Minutes)
48,Stand Group--141,2021-04-26 07:23:00,88.000000,0.00,67.000000,0.0
49,Stand Group--142,2021-04-26 11:42:00,111.000000,0.00,22.000000,0.0
50,Stand Group--143,2021-04-26 14:02:00,76.583333,56.00,43.666667,0.0
51,Stand Group--144,2021-04-26 17:01:40,76.500000,51.25,45.250000,0.0
52,Stand Group--145,2021-04-26 20:05:00,56.416667,49.25,49.500000,0.0
...,...,...,...,...,...,...
519,Stand Group--566,2021-05-09 12:51:09,0.000000,0.00,10.750000,0.0
520,Stand Group--567,2021-05-09 13:06:34,75.333333,0.00,25.500000,0.0
521,Stand Group--568,2021-05-09 14:51:09,89.583333,0.00,25.250000,0.0
522,Stand Group--569,2021-05-09 16:49:34,52.916667,0.00,115.716667,0.0


In [122]:
StandTimeBreakdown_DF = Data_Temp.groupby(['Stand Group','SUB-ACTIVITY']).agg({
            'Start Time':'first', 
            'ACTIVITY':'first', 
            'Duration (Minutes)':'sum',
            'Rotate Drilling Time (Minutes)':'sum', 
            'Slide Drilling Time (Minutes)':'sum',
            'Reaming Time (Minutes)':'sum',
            'Connection Time (Minutes)':'sum',
            }
        ).reset_index()
# StandTimeBreakdown_DF = StandTimeBreakdown_DF.loc[~(StandTimeBreakdown_DF[['Rotate Drilling Time (Minutes)','Slide Drilling Time (Minutes)','Reaming Time (Minutes)']]==0).all(axis=1)]
StandTimeBreakdown_DF

idx_logic_activity = StandTimeBreakdown_DF["ACTIVITY"].isin(["DRILLING FORMATION", 
                                            'CIRCULATE HOLE CLEANING',
                                            'CONNECTION',
                                            'DRILL OUT CEMENT',
                                        ])
StandTimeBreakdown_DF= StandTimeBreakdown_DF[idx_logic_activity]
# tes = 
StandTimeBreakdown_DF = StandTimeBreakdown_DF.join(StandTimeBreakdown_DF.groupby(['Stand Group']).agg({'Start Time':'first'}), on='Stand Group', rsuffix='_used')
# display(tes)
display(StandTimeBreakdown_DF)

figStandTimeBreakdown = px.bar(StandTimeBreakdown_DF, x='Start Time_used', y=['Duration (Minutes)'],color='SUB-ACTIVITY', width=1000, height=1000)
# figStandTimeBreakdown = px.bar(SummaryActivity_DF.dropna(subset=['Stand Group_Pred', "Duration(minutes)"]), x='Stand Group_Pred', y="Duration(minutes)", color="LABEL_SubActivity", width=1000, height=1000)
figStandTimeBreakdown.update_xaxes(type='category')
figStandTimeBreakdown.update_layout(
                # title="Title",
                xaxis=dict(
                    title="Stand Group"
                ),
                yaxis=dict(
                    title="Duration(Minutes)"
                ) ) 




,Stand Group,SUB-ACTIVITY,Start Time,ACTIVITY,Duration (Minutes),Rotate Drilling Time (Minutes),Slide Drilling Time (Minutes),Reaming Time (Minutes),Connection Time (Minutes),Start Time_used
48,Stand Group--139,Circulation,2021-04-26 05:51:00,DRILL OUT CEMENT,6.000000,0.000000,0.0,0.000000,0.0,2021-04-26 05:51:00
58,Stand Group--141,Rotary Drilling,2021-04-26 07:40:00,DRILL OUT CEMENT,88.000000,88.000000,0.0,0.000000,0.0,2021-04-26 07:40:00
59,Stand Group--142,Reaming,2021-04-26 11:42:00,DRILLING FORMATION,22.000000,0.000000,0.0,22.000000,0.0,2021-04-26 11:42:00
60,Stand Group--142,Rotary Drilling,2021-04-26 11:45:00,DRILLING FORMATION,111.000000,111.000000,0.0,0.000000,0.0,2021-04-26 11:42:00
61,Stand Group--143,Reaming,2021-04-26 14:02:00,DRILLING FORMATION,43.666667,0.000000,0.0,43.666667,0.0,2021-04-26 14:02:00
...,...,...,...,...,...,...,...,...,...,...
667,Stand Group--568,Reaming,2021-05-09 14:51:09,DRILLING FORMATION,25.250000,0.000000,0.0,25.250000,0.0,2021-05-09 14:51:09
668,Stand Group--568,Rotary Drilling,2021-05-09 14:54:54,DRILLING FORMATION,89.583333,89.583333,0.0,0.000000,0.0,2021-05-09 14:51:09
669,Stand Group--569,Reaming,2021-05-09 16:49:34,DRILLING FORMATION,115.716667,0.000000,0.0,115.716667,0.0,2021-05-09 16:49:34
670,Stand Group--569,Rotary Drilling,2021-05-09 16:53:49,DRILLING FORMATION,52.916667,52.916667,0.0,0.000000,0.0,2021-05-09 16:49:34
